# Build your own view

Companion notebook for the [Build your own view](https://ddacs.readthedocs.io/en/latest/tutorials/views/) tutorial. It is written to stand on its own: every step is annotated so the notebook reads top to bottom.

## Walkthrough

1. load the dataset and list the published RecordSets,
2. append a custom RecordSet with four representative field shapes,
3. inspect the resolved source field and timestep transforms,
4. iterate records of the new view and check their numpy shapes.

## Assumptions

- This notebook lives in `notebooks/` of the repository. The data directory `./data` therefore resolves to `../data` from here.
- `ddacs download --small` has been run once from the repository root, so `../data/` contains `metadata.json`, `process_parameters.csv`, and the bundled sample zip.

In [1]:
DATA_DIR = '../data'
sim_id   = 258864

import ddacs
print('ddacs', ddacs.__version__)

ddacs 3.0.0


## 1. Load the dataset

`ddacs.load(data_dir=...)` reads `metadata.json` from `data_dir`, registers any zip files it finds there so `mlcroissant` can read HDF5 members in place, and exposes the published RecordSets through `ds.metadata.record_sets`. Listing them first is good practice: a custom view is only ever needed when none of the published ones fit.

In [2]:
ds = ddacs.load(data_dir=DATA_DIR)
print('published RecordSets:')
for rs in ds.metadata.record_sets:
    print(f'  {rs.id}')

published RecordSets:
  process-parameters
  field-map
  simulation-provenance
  springback-minimal
  springback-prediction
  forming-snapshot
  cutting-view


## 2. Append a custom RecordSet

`ddacs.add_view(ds, name, fields)` mutates `ds` in place. After it returns, the new RecordSet is part of `ds.metadata.record_sets` and can be iterated with `ds.records(name)` exactly like a published one.

Each entry of `fields` maps an **alias** (the dict key) to a **source field** declared in the `field-map` RecordSet of the manifest, plus an optional **timestep selection**. The full list of source field IDs (e.g. `op10_blank_node_displacement`) lives in the [Complete field reference](https://ddacs.readthedocs.io/en/latest/hdf5-structure/#complete-field-reference) : that table is the lookup for what to put on the right-hand side of `fields`. The four entries below exercise every supported shape:

- `trajectory` keeps all timesteps -> the form for a sequence model that needs the full forming history.
- `forming` takes timestep `2` (max forming depth) -> the form for a single-snapshot model.
- `springback` takes timesteps `[2, 3]` (max forming and after springback) -> the form for a model whose target is the springback delta.
- `thickness` is the shortcut shape: a bare string means "the whole field", same as `("...", None)`.

The published manifest on DaRUS is **not** modified; only the in-memory dataset grows.

In [3]:
ddacs.add_view(
    ds,
    'my-view',
    fields={
        'trajectory': ('op10_blank_node_displacement', None),       # all 4 timesteps
        'forming':    ('op10_blank_node_displacement', 2),          # one timestep
        'springback': ('op10_blank_node_displacement', [2, 3]),     # subset of timesteps
        'thickness':  'op10_blank_element_shell_thickness',         # whole field, shortcut form
    },
)

print('record_sets after add_view:')
for rs in ds.metadata.record_sets:
    print(f'  {rs.id}')

record_sets after add_view:
  process-parameters
  field-map
  simulation-provenance
  springback-minimal
  springback-prediction
  forming-snapshot
  cutting-view
  my-view


## 3. Inspect the new view's fields

Each field of the view stores a **source** (the field-map entry it pulls from) and an **optional JSONPath transform** that records the timestep selection:

- `("field_id", 2)` -> JSONPath `$[2]`
- `("field_id", [2, 3])` -> JSONPath `$[2,3]`
- `"field_id"` or `("field_id", None)` -> no transform (whole field)

Printing the resolved transforms is a quick way to verify the view was built as expected before you start training on it.

In [4]:
view = next(rs for rs in ds.metadata.record_sets if rs.id == 'my-view')
for f in view.fields:
    transforms = [t.json_path for t in (f.source.transforms or [])]
    print(f'  {f.id:30s} <- {f.source.uuid:60s} transforms={transforms}')

  my-view/trajectory             <- field-map/op10_blank_node_displacement                       transforms=[]
  my-view/forming                <- field-map/op10_blank_node_displacement                       transforms=['$[2]']
  my-view/springback             <- field-map/op10_blank_node_displacement                       transforms=['$[2,3]']
  my-view/thickness              <- field-map/op10_blank_element_shell_thickness                 transforms=[]


## 4. Iterate records of the view

`ds.records('my-view')` is a generator over the simulations. Each record is a dict keyed by the aliases declared in `add_view`, with values already sliced according to the JSONPath transform. `forming` comes out as a 2D `(n_nodes, 3)` array (the `$[2]` transform dropped the timestep axis) while `trajectory` keeps the full `(4, n_nodes, 3)` shape.

> **Heads up.** `mlcroissant` walks every zip referenced by the FileSet and aborts at the first missing one. With the small bundle only `258864.zip` is on disk, so the cell below will error before it yields anything. The iteration works in full only after `ddacs download` (without `--small`) has pulled the complete release.
>
> If you need to iterate a partial download gracefully, `DDACSDataset` (PyTorch tutorial) builds a `sim_id -> local zip` index and silently skips simulations whose zip is missing.

In [5]:
import numpy as np
try:
    for rec in ds.records('my-view'):
        for k, v in rec.items():
            arr = np.asarray(v)
            print(f'  {k:20s} shape={arr.shape}  dtype={arr.dtype}')
        break
except Exception as e:
    print(f'records stopped: {type(e).__name__}: {str(e)[:200]}')

records stopped: GenerationError: An error occured during the sequential generation of the dataset, more specifically during the operation Download(106921_109120.zip)


## 5. Read other RecordSets

Two RecordSets ship with the published manifest that are useful even outside of view building:

- **`process-parameters`** is the simulation index : one record per simulation, with every column of `process_parameters.csv` exposed as a named field. It is sourced from the CSV (not from the h5 zips), so the records iterator works regardless of which zips are downloaded locally.
- **`simulation-provenance`** is a dataset-wide [SIM-KAx](https://doi.org/10.1007/s11740-026-01441-7) record describing the LS-DYNA setup (program, material model, mesh settings, contact, ...). Because its fields carry constant values directly in the manifest, they can be read off `ds.metadata` without iterating records at all.

### Iterating process-parameters

The pattern is identical to step 4: ask for `ds.records(<RecordSet id>)` and consume the iterator. The cell below prints the first row; for tabular work, `pandas.DataFrame(list(ds.records('process-parameters')))` is one line away, but the canonical access pattern is the iterator.

In [6]:
for n, rec in enumerate(ds.records('process-parameters'), start=1):
    if n == 1:
        for k, v in rec.items():
            print(f'  {k:42s} = {v}')
    if n >= 1:
        break

  process-parameters/index                   = 16039
  process-parameters/geometry                = b'rectangular'
  process-parameters/curvature_radius        = 30.0
  process-parameters/bottom_radius           = 5.0
  process-parameters/wall_angle              = 10.0
  process-parameters/material_scaling_factor = 0.9
  process-parameters/sheet_metal_thickness   = 0.95
  process-parameters/friction_coefficient    = 0.05
  process-parameters/blankholder_force       = 100000.0
  process-parameters/split                   = b'train'
  process-parameters/rddac                   = False


### Simulation provenance (SIM-KAx)

The `simulation-provenance` RecordSet carries the LS-DYNA setup constants. Its fields use the `value` form of mlcroissant's source spec, so the data lives directly in the manifest - no record iteration needed. Read the field list off `ds.metadata.record_sets`:

In [7]:
sp = next(rs for rs in ds.metadata.record_sets if rs.id == 'simulation-provenance')
for f in sp.fields:
    print(f'  {f.name:50s} = {f.value!r}')

  description                                        = 'Deep drawing of a modified quadratic cup (210 x 210 mm, 30 mm drawing depth) from DP600 dual-phase steel.'
  program                                            = 'LS-DYNA'
  tool_deformability                                 = 'rigid'
  tool_movement                                      = 'kinematic'
  model_symmetries                                   = 'none'
  meshing                                            = 'Fully integrated shell elements (ELFORM=16) with 7 integration points; planar regions ~10 mm edge, blank ~1 mm, finer at radii (>= 6 elements across).'
  element_size                                       = 1.0
  contact_tolerance                                  = 0.0
  contact_type                                       = '*CONTACT_FORMING_ONE_WAY_TO_SURFACE'
  material_model                                     = '*MAT_125 (Kinematic Hardening Transversely Anisotropic, Yoshida-Uemori, calibrated for DP600)'
  pre_stag